<a href="https://colab.research.google.com/github/fukagai-takuya/gifu-ai/blob/main/gifu-ai-2026-09-06/RagPoc_Population_by_Pref_Age_2026.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import requests
from pathlib import Path

# ==========================================
# e-Stat Excel ダウンロード
# ==========================================

URL = "https://www.e-stat.go.jp/stat-search/file-download?statInfId=000040479048&fileKind=0"

OUTPUT = Path(
    "/content/rag-poc/data/excel/Population_by_Pref_Age_2026.xlsx"
)

OUTPUT.parent.mkdir(parents=True, exist_ok=True)

response = requests.get(URL)
response.raise_for_status()

with open(OUTPUT, "wb") as f:
    f.write(response.content)

print(f"ダウンロード完了")
print(f"保存先: {OUTPUT}")
print(f"ファイルサイズ: {OUTPUT.stat().st_size:,} bytes")

ダウンロード完了
保存先: /content/rag-poc/data/excel/Population_by_Pref_Age_2026.xlsx
ファイルサイズ: 46,931 bytes


In [2]:
import openpyxl

path = "/content/rag-poc/data/excel/Population_by_Pref_Age_2026.xlsx"

wb = openpyxl.load_workbook(path, read_only=True, data_only=True)

print("Excel読み込み成功")
print("Sheet:", wb.sheetnames)

Excel読み込み成功
Sheet: ['年齢別人口（都道府県別）【総計】']


In [3]:
# ==========================================
# RAG PoC 環境構築
# ==========================================

!pip install -q \
    openpyxl \
    sentence-transformers \
    qdrant-client \
    transformers \
    accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 396.2/396.2 kB 25.1 MB/s eta 0:00:00


In [4]:
# ==========================================
# RAG PoC ディレクトリ作成
# ==========================================

from pathlib import Path

BASE_DIR = Path("/content/rag-poc")
DATA_DIR = BASE_DIR / "data" / "excel"
QDRANT_DIR = BASE_DIR / "qdrant_data"

DATA_DIR.mkdir(parents=True, exist_ok=True)
QDRANT_DIR.mkdir(parents=True, exist_ok=True)

print(f"BASE_DIR   : {BASE_DIR}")
print(f"DATA_DIR   : {DATA_DIR}")
print(f"QDRANT_DIR : {QDRANT_DIR}")

BASE_DIR   : /content/rag-poc
DATA_DIR   : /content/rag-poc/data/excel
QDRANT_DIR : /content/rag-poc/qdrant_data


In [5]:
# ==========================================
# Excel → JSON
# ==========================================

import json
import openpyxl

INPUT = "/content/rag-poc/data/excel/Population_by_Pref_Age_2026.xlsx"
OUTPUT = "/content/rag-poc/data/excel/Population_by_Pref_Age_2026.json"

wb = openpyxl.load_workbook(INPUT, data_only=True)
ws = wb.active

# 1行目：タイトル
title = ws.cell(1, 1).value

# 2行目：年齢階級
age_headers = [cell.value for cell in ws[2][3:]]

# 3行目：単位
units = [cell.value for cell in ws[3][3:]]

# 4～147行目：データ
records = []

for row in ws.iter_rows(min_row=4, max_row=147, values_only=True):
    record = {
        "団体コード": row[0],
        "都道府県名": row[1],
        "性別": row[2],
    }

    for age, value, unit in zip(age_headers, row[3:], units):
        record[age] = {
            "value": value,
            "unit": unit,
        }

    records.append(record)

# 148～150行目：注記
notes = [
    ws.cell(row, 1).value
    for row in range(148, 151)
    if ws.cell(row, 1).value
]

data = {
    "title": title,
    "source": INPUT,
    "notes": notes,
    "records": records,
}

with open(OUTPUT, "w", encoding="utf-8") as f:
    json.dump(data, f, ensure_ascii=False, indent=2)

print(f"入力: {INPUT}")
print(f"出力: {OUTPUT}")
print(f"レコード数: {len(records)}")
print(f"注記数: {len(notes)}")

入力: /content/rag-poc/data/excel/Population_by_Pref_Age_2026.xlsx
出力: /content/rag-poc/data/excel/Population_by_Pref_Age_2026.json
レコード数: 144
注記数: 3


In [6]:
# ==========================================
# JSON → Chunk
# ==========================================

import json

INPUT = "/content/rag-poc/data/excel/Population_by_Pref_Age_2026.json"
OUTPUT = "/content/rag-poc/data/excel/Population_by_Pref_Age_2026-chunks.json"

with open(INPUT, "r", encoding="utf-8") as f:
    data = json.load(f)

chunks = []

for chunk_id, record in enumerate(data["records"]):
    lines = [
        data["title"],
        "",
        f"都道府県名：{record['都道府県名']}",
        f"性別：{record['性別']}",
        "",
    ]

    for key, item in record.items():
        if key in ["団体コード", "都道府県名", "性別"]:
            continue

        lines.append(
            f"{key}：{item['value']:,}{item['unit']}"
        )

    text = "\n".join(lines)

    chunks.append({
        "chunk_id": chunk_id,
        "text": text,
        "metadata": {
            "source": data["source"],
            "title": data["title"],
            "団体コード": record["団体コード"],
            "都道府県名": record["都道府県名"],
            "性別": record["性別"],
        }
    })

# 注記もChunkとして追加
for note in data["notes"]:
    chunk_id = len(chunks)

    text = f"{data['title']}\n\n注記：\n{note}"

    chunks.append({
        "chunk_id": chunk_id,
        "text": text,
        "metadata": {
            "source": data["source"],
            "title": data["title"],
            "type": "note"
        }
    })

with open(OUTPUT, "w", encoding="utf-8") as f:
    json.dump(chunks, f, ensure_ascii=False, indent=2)

print(f"入力: {INPUT}")
print(f"出力: {OUTPUT}")
print(f"Chunk数: {len(chunks)}")

入力: /content/rag-poc/data/excel/Population_by_Pref_Age_2026.json
出力: /content/rag-poc/data/excel/Population_by_Pref_Age_2026-chunks.json
Chunk数: 147


In [7]:
# ==========================================
# BGE-M3 Embedding
# ==========================================

import json
from sentence_transformers import SentenceTransformer

INPUT = "/content/rag-poc/data/excel/Population_by_Pref_Age_2026-chunks.json"
OUTPUT = "/content/rag-poc/data/excel/Population_by_Pref_Age_2026-embeddings.json"

MODEL_NAME = "BAAI/bge-m3"

# BGE-M3をGPUで読み込み
print("BGE-M3を読み込んでいます...")

embed_model = SentenceTransformer(
    MODEL_NAME,
    device="cuda",
)

# Chunk読み込み
with open(INPUT, "r", encoding="utf-8") as f:
    chunks = json.load(f)

texts = [chunk["text"] for chunk in chunks]

print(f"Chunk数: {len(texts)}")

# Embedding
embeddings = embed_model.encode(
    texts,
    batch_size=8,
    show_progress_bar=True,
    normalize_embeddings=True,
)

# 保存
results = []

for chunk, embedding in zip(chunks, embeddings):
    results.append({
        "chunk_id": chunk["chunk_id"],
        "text": chunk["text"],
        "metadata": chunk["metadata"],
        "embedding": embedding.tolist(),
    })

with open(OUTPUT, "w", encoding="utf-8") as f:
    json.dump(
        results,
        f,
        ensure_ascii=False,
        indent=2,
    )

print(f"入力: {INPUT}")
print(f"出力: {OUTPUT}")
print(f"Chunk数: {len(results)}")
print(f"Embedding次元数: {len(results[0]['embedding'])}")

BGE-M3を読み込んでいます...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/15.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

Chunk数: 147


Batches:   0%|          | 0/19 [00:00<?, ?it/s]

入力: /content/rag-poc/data/excel/Population_by_Pref_Age_2026-chunks.json
出力: /content/rag-poc/data/excel/Population_by_Pref_Age_2026-embeddings.json
Chunk数: 147
Embedding次元数: 1024


In [9]:
# ==========================================
# BGE-M3解放
# ==========================================

import gc
import torch

del embed_model

gc.collect()
torch.cuda.empty_cache()

print("BGE-M3を解放しました。")
print(f"GPUメモリ使用量: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")

BGE-M3を解放しました。
GPUメモリ使用量: 0.01 GB


In [10]:
# ==========================================
# GPUメモリ確認
# ==========================================

!nvidia-smi

Fri Aug 21 05:23:40 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   61C    P0             29W /   70W |     153MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [11]:
# ==========================================
# Qdrant Collection作成・登録
# ==========================================

import json
from pathlib import Path

from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct

EMBEDDINGS_FILE = Path(
    "/content/rag-poc/data/excel/Population_by_Pref_Age_2026-embeddings.json"
)

COLLECTION_NAME = "population_age_2026"
VECTOR_SIZE = 1024

QDRANT_DIR = "/content/rag-poc/qdrant_data"

# Qdrant Local
client = QdrantClient(
    path=QDRANT_DIR
)

# Embeddings JSON読み込み
with open(EMBEDDINGS_FILE, "r", encoding="utf-8") as f:
    chunks = json.load(f)

print(f"読み込んだChunk数: {len(chunks)}")

# Collection作成
if client.collection_exists(COLLECTION_NAME):

    print(
        f"Collection '{COLLECTION_NAME}' は既に存在します。"
    )

else:

    client.create_collection(
        collection_name=COLLECTION_NAME,
        vectors_config=VectorParams(
            size=VECTOR_SIZE,
            distance=Distance.COSINE,
        ),
    )

    print(
        f"Collection '{COLLECTION_NAME}' を作成しました。"
    )

# Qdrant Point作成
points = []

for chunk in chunks:

    points.append(
        PointStruct(
            id=chunk["chunk_id"],
            vector=chunk["embedding"],
            payload={
                "text": chunk["text"],
                "metadata": chunk["metadata"],
            },
        )
    )

# Qdrantへ登録
client.upsert(
    collection_name=COLLECTION_NAME,
    points=points,
)

print(
    f"{len(points)}個のChunkをQdrantへ登録しました。"
)

# 登録結果確認
collection_info = client.get_collection(
    collection_name=COLLECTION_NAME
)

print()
print(f"Collection: {COLLECTION_NAME}")
print(
    f"登録されたVector数: "
    f"{collection_info.points_count}"
)

読み込んだChunk数: 147
Collection 'population_age_2026' を作成しました。
147個のChunkをQdrantへ登録しました。

Collection: population_age_2026
登録されたVector数: 147


In [12]:
# ==========================================
# Qdrantベクトル検索
# ==========================================

import numpy as np
from sentence_transformers import SentenceTransformer

EMBED_MODEL = "BAAI/bge-m3"

QUERIES = [
    "北海道の65歳～69歳の人口は？",
    "北海道の65歳～69歳の男性人口は？",
    "北海道の20歳～24歳の女性人口は？",
    "岐阜県の75歳～79歳の人口は？",
    "東京都の100歳以上の女性人口は？",
]

TOP_K = 5

# BGE-M3
print("BGE-M3を読み込んでいます...")

embed_model = SentenceTransformer(
    EMBED_MODEL,
    device="cuda",
)

# QueryをまとめてEmbedding
query_embeddings = embed_model.encode(
    QUERIES,
    normalize_embeddings=True,
)

# QueryごとにQdrant検索
for query, query_embedding in zip(
    QUERIES,
    query_embeddings,
):

    results = client.query_points(
        collection_name=COLLECTION_NAME,
        query=query_embedding.tolist(),
        limit=TOP_K,
        with_payload=True,
    ).points

    print()
    print("=" * 80)
    print(f"Query: {query}")
    print("=" * 80)

    for rank, result in enumerate(results, start=1):

        metadata = result.payload.get(
            "metadata",
            {}
        )

        print(
            f"[{rank}] "
            f"score={result.score:.4f} "
            f"chunk_id={result.id} "
            f"{metadata.get('都道府県名')} "
            f"{metadata.get('性別')}"
        )

BGE-M3を読み込んでいます...


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]


Query: 北海道の65歳～69歳の人口は？
[1] score=0.6676 chunk_id=5 北海道 女
[2] score=0.6583 chunk_id=3 北海道 計
[3] score=0.6579 chunk_id=4 北海道 男
[4] score=0.5571 chunk_id=93 鳥取県 計
[5] score=0.5514 chunk_id=2 合計 女

Query: 北海道の65歳～69歳の男性人口は？
[1] score=0.6759 chunk_id=4 北海道 男
[2] score=0.6540 chunk_id=5 北海道 女
[3] score=0.6512 chunk_id=3 北海道 計
[4] score=0.5665 chunk_id=1 合計 男
[5] score=0.5657 chunk_id=94 鳥取県 男

Query: 北海道の20歳～24歳の女性人口は？
[1] score=0.6209 chunk_id=5 北海道 女
[2] score=0.5965 chunk_id=3 北海道 計
[3] score=0.5764 chunk_id=4 北海道 男
[4] score=0.5050 chunk_id=11 岩手県 女
[5] score=0.4975 chunk_id=62 長野県 女

Query: 岐阜県の75歳～79歳の人口は？
[1] score=0.6621 chunk_id=63 岐阜県 計
[2] score=0.6569 chunk_id=65 岐阜県 女
[3] score=0.6522 chunk_id=64 岐阜県 男
[4] score=0.5462 chunk_id=120 福岡県 計
[5] score=0.5371 chunk_id=86 兵庫県 女

Query: 東京都の100歳以上の女性人口は？
[1] score=0.6847 chunk_id=41 東京都 女
[2] score=0.6589 chunk_id=39 東京都 計
[3] score=0.6363 chunk_id=40 東京都 男
[4] score=0.5890 chunk_id=35 埼玉県 女
[5] score=0.5866 chunk_id=80 京都府 女


In [13]:
# ==========================================
# BGE-M3解放
# ==========================================

import gc
import torch

del embed_model

gc.collect()
torch.cuda.empty_cache()

print("BGE-M3を解放しました。")
print(
    f"GPUメモリ使用量: "
    f"{torch.cuda.memory_allocated() / 1024**3:.2f} GB"
)

BGE-M3を解放しました。
GPUメモリ使用量: 0.01 GB


In [14]:
# ==========================================
# Qwen実行環境
# ==========================================

!pip install -q bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.0/41.0 MB 22.5 MB/s eta 0:00:00


In [15]:
# ==========================================
# Qwen3-8B読み込み
# ==========================================

import torch

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
)

LLM_MODEL = "Qwen/Qwen3-8B"

# 4bit量子化
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)

print("Qwen Tokenizerを読み込んでいます...")

tokenizer = AutoTokenizer.from_pretrained(
    LLM_MODEL,
)

print("Qwenモデルを読み込んでいます...")

model = AutoModelForCausalLM.from_pretrained(
    LLM_MODEL,
    quantization_config=quantization_config,
    device_map="auto",
)

print("Qwenの読み込みが完了しました。")

print(
    f"GPUメモリ使用量: "
    f"{torch.cuda.memory_allocated() / 1024**3:.2f} GB"
)

Qwen Tokenizerを読み込んでいます...


config.json:   0%|          | 0.00/728 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.73k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/399 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

Qwenの読み込みが完了しました。
GPUメモリ使用量: 5.98 GB


In [16]:
# ==========================================
# RAG回答テスト
# ==========================================

QUERY = "北海道の65歳～69歳の人口は？"

TOP_K = 3

# BGE-M3をCPUで読み込み
# Qwen用のGPUメモリを確保するためCPUで実行
print("BGE-M3を読み込んでいます...")

embed_model = SentenceTransformer(
    "BAAI/bge-m3",
    device="cpu",
)

# Query Embedding
query_vector = embed_model.encode(
    QUERY,
    normalize_embeddings=True,
)

# Qdrant検索
results = client.query_points(
    collection_name=COLLECTION_NAME,
    query=query_vector.tolist(),
    limit=TOP_K,
    with_payload=True,
).points

# Context作成
contexts = []

for i, result in enumerate(results, start=1):

    payload = result.payload
    metadata = payload.get("metadata", {})

    context = f"""
[検索結果 {i}]
都道府県名：{metadata.get('都道府県名')}
性別：{metadata.get('性別')}

{payload.get('text')}
"""

    contexts.append(context)

context_text = "\n\n---\n\n".join(contexts)

# 検索結果確認
print()
print("=" * 80)
print("検索結果")
print("=" * 80)

for i, result in enumerate(results, start=1):

    metadata = result.payload.get(
        "metadata",
        {}
    )

    print(
        f"[{i}] "
        f"{metadata.get('都道府県名')} "
        f"{metadata.get('性別')} "
        f"(score={result.score:.4f})"
    )

BGE-M3を読み込んでいます...


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]


検索結果
[1] 北海道 女 (score=0.6676)
[2] 北海道 計 (score=0.6583)
[3] 北海道 男 (score=0.6579)


In [18]:
print(context_text)


[検索結果 1]
都道府県名：北海道
性別：女

令和8年1月1日住民基本台帳年齢階級別人口（都道府県別）（総計）

都道府県名：北海道
性別：女

総数：2,630,860人
0歳～4歳：60,955人
5歳～9歳：80,115人
10歳～14歳：93,351人
15歳～19歳：100,335人
20歳～24歳：109,554人
25歳～29歳：108,341人
30歳～34歳：113,641人
35歳～39歳：125,358人
40歳～44歳：146,894人
45歳～49歳：169,686人
50歳～54歳：197,204人
55歳～59歳：179,532人
60歳～64歳：177,945人
65歳～69歳：171,549人
70歳～74歳：199,868人
75歳～79歳：219,726人
80歳～84歳：157,081人
85歳～89歳：118,113人
90歳～94歳：71,272人
95歳～99歳：25,278人
100歳以上：4,408人


---


[検索結果 2]
都道府県名：北海道
性別：計

令和8年1月1日住民基本台帳年齢階級別人口（都道府県別）（総計）

都道府県名：北海道
性別：計

総数：4,996,492人
0歳～4歳：124,861人
5歳～9歳：164,196人
10歳～14歳：191,197人
15歳～19歳：206,511人
20歳～24歳：225,377人
25歳～29歳：224,674人
30歳～34歳：231,977人
35歳～39歳：254,123人
40歳～44歳：293,279人
45歳～49歳：338,125人
50歳～54歳：391,295人
55歳～59歳：347,479人
60歳～64歳：340,218人
65歳～69歳：327,278人
70歳～74歳：370,277人
75歳～79歳：388,205人
80歳～84歳：257,923人
85歳～89歳：181,358人
90歳～94歳：100,067人
95歳～99歳：31,737人
100歳以上：5,077人


---


[検索結果 3]
都道府県名：北海道
性別：男

令和8年1月1日住民基本台帳年齢階級別人口（都道府県別）（総計）

都道府県名：北海道
性別：男

総数：2,365,632人
0歳～4歳：63,906人
5歳～9歳：84

In [19]:
# ==========================================
# RAG System Prompt
# ==========================================

system_prompt = """
あなたは日本の人口統計データを回答するRAGアシスタントです。

必ずコンテキストに記載されたデータだけを使って回答してください。
コンテキストにない数値を推測・計算・創作してはいけません。

【回答するデータの選択】

質問から以下の3項目を特定してください。

1. 都道府県
2. 年齢階級
3. 性別

性別については次のルールに従ってください。

・「男性」「男」→ 性別：男
・「女性」「女」→ 性別：女
・性別の指定なし → 性別：計

性別の指定がない場合は、必ず「性別：計」を使用してください。
「男」や「女」のデータを合計してはいけません。

【重要】

都道府県、年齢階級、性別のすべてが一致するデータを
コンテキストから探してください。

一致するデータが複数ある場合は、
質問の条件に最も一致するデータを使用してください。

該当するデータがコンテキストにない場合は、
「該当するデータが見つかりませんでした。」
と回答してください。

【回答形式】

該当する人口を簡潔に回答してください。
"""

In [20]:
# ==========================================
# Qwen RAG回答
# ==========================================

user_prompt = f"""
### 質問

{QUERY}

### コンテキスト

{context_text}

### 回答

質問に対して簡潔に回答してください。
"""

messages = [
    {
        "role": "system",
        "content": system_prompt,
    },
    {
        "role": "user",
        "content": user_prompt,
    },
]

# Chat Template
text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=False,
)

inputs = tokenizer(
    text,
    return_tensors="pt",
).to(model.device)

input_tokens = inputs["input_ids"].shape[1]

print()
print("=" * 80)
print("Qwen3-8B 回答生成")
print("=" * 80)

print(f"入力トークン数: {input_tokens}")

# 生成
with torch.no_grad():

    output_ids = model.generate(
        **inputs,
        max_new_tokens=512,
        do_sample=False,
    )

# 入力部分を除いて回答部分だけ取得
generated_ids = output_ids[0][input_tokens:]

answer = tokenizer.decode(
    generated_ids,
    skip_special_tokens=True,
)

output_tokens = len(generated_ids)
total_tokens = input_tokens + output_tokens

print()
print("=" * 80)
print("Token使用量")
print("=" * 80)

print(f"入力トークン数 : {input_tokens}")
print(f"生成トークン数 : {output_tokens}")
print(f"合計Token数    : {total_tokens}")

print()
print("=" * 80)
print("Qwen3-8Bの回答")
print("=" * 80)

print(answer)


Qwen3-8B 回答生成
入力トークン数: 1621

Token使用量
入力トークン数 : 1621
生成トークン数 : 24
合計Token数    : 1645

Qwen3-8Bの回答
北海道の65歳～69歳の人口は327,278人です。


In [21]:
# ==========================================
# Qwen RAG回答テスト
# ==========================================

import torch
from sentence_transformers import SentenceTransformer


# -----------------------------
# 設定
# -----------------------------

EMBED_MODEL = "BAAI/bge-m3"

TOP_K = 3

CONTEXT_LENGTH = 8192

QUERIES = [
    "北海道の65歳～69歳の人口は？",
    "北海道の65歳～69歳の男性人口は？",
    "北海道の20歳～24歳の女性人口は？",
    "岐阜県の75歳～79歳の人口は？",
    "東京都の100歳以上の女性人口は？",
]


# -----------------------------
# BGE-M3
# -----------------------------

print("BGE-M3を読み込んでいます...")

embed_model = SentenceTransformer(
    EMBED_MODEL,
    device="cpu",
)

print("BGE-M3の読み込みが完了しました。")


# ==========================================================
# 質問ごとにRAG回答
# ==========================================================

for query_number, QUERY in enumerate(QUERIES, start=1):

    print()
    print()
    print("#" * 80)
    print(f"質問 {query_number}")
    print("#" * 80)

    print()
    print(f"質問: {QUERY}")


    # ======================================================
    # Query Embedding
    # ======================================================

    query_vector = embed_model.encode(
        QUERY,
        normalize_embeddings=True,
    )


    # ======================================================
    # Qdrant検索
    # ======================================================

    results = client.query_points(
        collection_name=COLLECTION_NAME,
        query=query_vector.tolist(),
        limit=TOP_K,
        with_payload=True,
    ).points


    # ======================================================
    # 検索結果表示
    # ======================================================

    print()
    print("=" * 80)
    print("検索結果")
    print("=" * 80)

    for i, result in enumerate(results, start=1):

        metadata = result.payload.get(
            "metadata",
            {}
        )

        print(
            f"[{i}] "
            f"score={result.score:.4f} "
            f"{metadata.get('都道府県名')} "
            f"{metadata.get('性別')}"
        )


    # ======================================================
    # Context作成
    # ======================================================

    contexts = []

    for i, result in enumerate(results, start=1):

        payload = result.payload
        metadata = payload.get("metadata", {})

        context = f"""
[検索結果 {i}]
都道府県名：{metadata.get('都道府県名')}
性別：{metadata.get('性別')}

{payload.get('text')}
"""

        contexts.append(context)


    context_text = "\n\n---\n\n".join(contexts)


    # ======================================================
    # User Prompt
    # ======================================================

    user_prompt = f"""
### 質問

{QUERY}

### コンテキスト

{context_text}

### 回答

質問に対して簡潔に回答してください。
"""


    # ======================================================
    # Messages
    # ======================================================

    messages = [
        {
            "role": "system",
            "content": system_prompt,
        },
        {
            "role": "user",
            "content": user_prompt,
        },
    ]


    # ======================================================
    # Chat Template
    # ======================================================

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False,
    )


    # ======================================================
    # Tokenize
    # ======================================================

    inputs = tokenizer(
        text,
        return_tensors="pt",
    )

    inputs = {
        key: value.to(model.device)
        for key, value in inputs.items()
    }

    input_tokens = inputs["input_ids"].shape[1]


    # ======================================================
    # Context Length
    # ======================================================

    remaining_tokens = CONTEXT_LENGTH - input_tokens

    usage_percent = (
        input_tokens / CONTEXT_LENGTH * 100
    )


    print()
    print("=" * 80)
    print("Context Length")
    print("=" * 80)

    print(f"入力トークン数 : {input_tokens}")
    print(f"Context Length : {CONTEXT_LENGTH}")
    print(f"残り          : {remaining_tokens}")
    print(f"使用率        : {usage_percent:.1f}%")


    # ======================================================
    # Qwen3-8B生成
    # ======================================================

    print()
    print("=" * 80)
    print("Qwen3-8B 回答生成")
    print("=" * 80)

    with torch.no_grad():

        output_ids = model.generate(
            **inputs,
            max_new_tokens=512,
            do_sample=False,
        )


    # ======================================================
    # 回答取得
    # ======================================================

    generated_ids = output_ids[0][input_tokens:]

    answer = tokenizer.decode(
        generated_ids,
        skip_special_tokens=True,
    )

    output_tokens = len(generated_ids)

    total_tokens = (
        input_tokens
        + output_tokens
    )


    # ======================================================
    # Token使用量
    # ======================================================

    print()
    print("=" * 80)
    print("Token使用量")
    print("=" * 80)

    print(f"入力トークン数 : {input_tokens}")
    print(f"生成トークン数 : {output_tokens}")
    print(f"合計Token数    : {total_tokens}")
    print(f"Context Length : {CONTEXT_LENGTH}")


    # ======================================================
    # Context使用率
    # ======================================================

    usage_percent_total = (
        total_tokens / CONTEXT_LENGTH * 100
    )

    remaining_tokens_total = (
        CONTEXT_LENGTH - total_tokens
    )

    print(f"Context残り    : {remaining_tokens_total}")
    print(f"Context使用率  : {usage_percent_total:.1f}%")


    # ======================================================
    # 回答
    # ======================================================

    print()
    print("=" * 80)
    print("Qwen3-8Bの回答")
    print("=" * 80)

    print(answer)

BGE-M3を読み込んでいます...


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

BGE-M3の読み込みが完了しました。


################################################################################
質問 1
################################################################################

質問: 北海道の65歳～69歳の人口は？

検索結果
[1] score=0.6676 北海道 女
[2] score=0.6583 北海道 計
[3] score=0.6579 北海道 男

Context Length
入力トークン数 : 1621
Context Length : 8192
残り          : 6571
使用率        : 19.8%

Qwen3-8B 回答生成

Token使用量
入力トークン数 : 1621
生成トークン数 : 24
合計Token数    : 1645
Context Length : 8192
Context残り    : 6547
Context使用率  : 20.1%

Qwen3-8Bの回答
北海道の65歳～69歳の人口は327,278人です。


################################################################################
質問 2
################################################################################

質問: 北海道の65歳～69歳の男性人口は？

検索結果
[1] score=0.6759 北海道 男
[2] score=0.6540 北海道 女
[3] score=0.6512 北海道 計

Context Length
入力トークン数 : 1622
Context Length : 8192
残り          : 6570
使用率        : 19.8%

Qwen3-8B 回答生成

Token使用量
入力トークン数 : 1622
生成トークン数 : 25
合計Token数    : 1647
Context Length : 8

In [22]:
# ==========================================
# Qwen RAG 失敗・境界条件テスト
# ==========================================

QUERIES = [
    "北海道の61歳～64歳の人口は？",
    "沖縄県の65歳～69歳の人口は？",
    "北海道の150歳～159歳の人口は？",
    "北海道の65歳～69歳の男女別人口は？",
    "北海道の人口は？",
]


# ==========================================================
# 質問ごとにRAG回答
# ==========================================================

for query_number, QUERY in enumerate(QUERIES, start=1):

    print()
    print()
    print("#" * 80)
    print(f"質問 {query_number}")
    print("#" * 80)

    print()
    print(f"質問: {QUERY}")


    # ======================================================
    # Query Embedding
    # ======================================================

    query_vector = embed_model.encode(
        QUERY,
        normalize_embeddings=True,
    )


    # ======================================================
    # Qdrant検索
    # ======================================================

    results = client.query_points(
        collection_name=COLLECTION_NAME,
        query=query_vector.tolist(),
        limit=TOP_K,
        with_payload=True,
    ).points


    # ======================================================
    # 検索結果
    # ======================================================

    print()
    print("=" * 80)
    print("検索結果")
    print("=" * 80)

    for i, result in enumerate(results, start=1):

        metadata = result.payload.get(
            "metadata",
            {}
        )

        print(
            f"[{i}] "
            f"score={result.score:.4f} "
            f"{metadata.get('都道府県名')} "
            f"{metadata.get('性別')}"
        )


    # ======================================================
    # Context作成
    # ======================================================

    contexts = []

    for i, result in enumerate(results, start=1):

        payload = result.payload
        metadata = payload.get("metadata", {})

        context = f"""
[検索結果 {i}]
都道府県名：{metadata.get('都道府県名')}
性別：{metadata.get('性別')}

{payload.get('text')}
"""

        contexts.append(context)


    context_text = "\n\n---\n\n".join(contexts)


    # ======================================================
    # User Prompt
    # ======================================================

    user_prompt = f"""
### 質問

{QUERY}

### コンテキスト

{context_text}

### 回答

質問に対して簡潔に回答してください。
"""


    # ======================================================
    # Messages
    # ======================================================

    messages = [
        {
            "role": "system",
            "content": system_prompt,
        },
        {
            "role": "user",
            "content": user_prompt,
        },
    ]


    # ======================================================
    # Chat Template
    # ======================================================

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False,
    )


    # ======================================================
    # Tokenize
    # ======================================================

    inputs = tokenizer(
        text,
        return_tensors="pt",
    )

    inputs = {
        key: value.to(model.device)
        for key, value in inputs.items()
    }

    input_tokens = inputs["input_ids"].shape[1]


    # ======================================================
    # Context Length
    # ======================================================

    remaining_tokens = (
        CONTEXT_LENGTH - input_tokens
    )

    usage_percent = (
        input_tokens
        / CONTEXT_LENGTH
        * 100
    )

    print()
    print("=" * 80)
    print("Context Length")
    print("=" * 80)

    print(f"入力トークン数 : {input_tokens}")
    print(f"Context Length : {CONTEXT_LENGTH}")
    print(f"残り          : {remaining_tokens}")
    print(f"使用率        : {usage_percent:.1f}%")


    # ======================================================
    # Qwen3-8B生成
    # ======================================================

    print()
    print("=" * 80)
    print("Qwen3-8B 回答生成")
    print("=" * 80)

    with torch.no_grad():

        output_ids = model.generate(
            **inputs,
            max_new_tokens=512,
            do_sample=False,
        )


    # ======================================================
    # 回答取得
    # ======================================================

    generated_ids = output_ids[0][input_tokens:]

    answer = tokenizer.decode(
        generated_ids,
        skip_special_tokens=True,
    )

    output_tokens = len(generated_ids)

    total_tokens = (
        input_tokens
        + output_tokens
    )


    # ======================================================
    # Token使用量
    # ======================================================

    print()
    print("=" * 80)
    print("Token使用量")
    print("=" * 80)

    print(f"入力トークン数 : {input_tokens}")
    print(f"生成トークン数 : {output_tokens}")
    print(f"合計Token数    : {total_tokens}")
    print(f"Context Length : {CONTEXT_LENGTH}")

    remaining_tokens_total = (
        CONTEXT_LENGTH - total_tokens
    )

    usage_percent_total = (
        total_tokens
        / CONTEXT_LENGTH
        * 100
    )

    print(f"Context残り    : {remaining_tokens_total}")
    print(f"Context使用率  : {usage_percent_total:.1f}%")


    # ======================================================
    # 回答
    # ======================================================

    print()
    print("=" * 80)
    print("Qwen3-8Bの回答")
    print("=" * 80)

    print(answer)



################################################################################
質問 1
################################################################################

質問: 北海道の61歳～64歳の人口は？

検索結果
[1] score=0.6285 北海道 女
[2] score=0.6136 北海道 計
[3] score=0.6124 北海道 男

Context Length
入力トークン数 : 1621
Context Length : 8192
残り          : 6571
使用率        : 19.8%

Qwen3-8B 回答生成

Token使用量
入力トークン数 : 1621
生成トークン数 : 24
合計Token数    : 1645
Context Length : 8192
Context残り    : 6547
Context使用率  : 20.1%

Qwen3-8Bの回答
北海道の61歳～64歳の人口は177,945人です。


################################################################################
質問 2
################################################################################

質問: 沖縄県の65歳～69歳の人口は？

検索結果
[1] score=0.6880 沖縄県 男
[2] score=0.6878 沖縄県 女
[3] score=0.6857 沖縄県 計

Context Length
入力トークン数 : 1582
Context Length : 8192
残り          : 6610
使用率        : 19.3%

Qwen3-8B 回答生成

Token使用量
入力トークン数 : 1582
生成トークン数 : 20
合計Token数    : 1602
Context Length : 8192
Context残り    : 659

In [23]:
# ==========================================
# Qwen RAG回答テスト (enable_thinking=True)
# ==========================================

import torch
from sentence_transformers import SentenceTransformer


# -----------------------------
# 設定
# -----------------------------

EMBED_MODEL = "BAAI/bge-m3"

TOP_K = 3

CONTEXT_LENGTH = 8192

QUERIES = [
    "北海道の65歳～69歳の人口は？",
    "北海道の65歳～69歳の男性人口は？",
    "北海道の20歳～24歳の女性人口は？",
    "岐阜県の75歳～79歳の人口は？",
    "東京都の100歳以上の女性人口は？",
]


# -----------------------------
# BGE-M3
# -----------------------------

print("BGE-M3を読み込んでいます...")

embed_model = SentenceTransformer(
    EMBED_MODEL,
    device="cpu",
)

print("BGE-M3の読み込みが完了しました。")


# ==========================================================
# 質問ごとにRAG回答
# ==========================================================

for query_number, QUERY in enumerate(QUERIES, start=1):

    print()
    print()
    print("#" * 80)
    print(f"質問 {query_number}")
    print("#" * 80)

    print()
    print(f"質問: {QUERY}")


    # ======================================================
    # Query Embedding
    # ======================================================

    query_vector = embed_model.encode(
        QUERY,
        normalize_embeddings=True,
    )


    # ======================================================
    # Qdrant検索
    # ======================================================

    results = client.query_points(
        collection_name=COLLECTION_NAME,
        query=query_vector.tolist(),
        limit=TOP_K,
        with_payload=True,
    ).points


    # ======================================================
    # 検索結果表示
    # ======================================================

    print()
    print("=" * 80)
    print("検索結果")
    print("=" * 80)

    for i, result in enumerate(results, start=1):

        metadata = result.payload.get(
            "metadata",
            {}
        )

        print(
            f"[{i}] "
            f"score={result.score:.4f} "
            f"{metadata.get('都道府県名')} "
            f"{metadata.get('性別')}"
        )


    # ======================================================
    # Context作成
    # ======================================================

    contexts = []

    for i, result in enumerate(results, start=1):

        payload = result.payload
        metadata = payload.get("metadata", {})

        context = f"""
[検索結果 {i}]
都道府県名：{metadata.get('都道府県名')}
性別：{metadata.get('性別')}

{payload.get('text')}
"""

        contexts.append(context)


    context_text = "\n\n---\n\n".join(contexts)


    # ======================================================
    # User Prompt
    # ======================================================

    user_prompt = f"""
### 質問

{QUERY}

### コンテキスト

{context_text}

### 回答

質問に対して簡潔に回答してください。
"""


    # ======================================================
    # Messages
    # ======================================================

    messages = [
        {
            "role": "system",
            "content": system_prompt,
        },
        {
            "role": "user",
            "content": user_prompt,
        },
    ]


    # ======================================================
    # Chat Template
    # ======================================================

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=True,
    )


    # ======================================================
    # Tokenize
    # ======================================================

    inputs = tokenizer(
        text,
        return_tensors="pt",
    )

    inputs = {
        key: value.to(model.device)
        for key, value in inputs.items()
    }

    input_tokens = inputs["input_ids"].shape[1]


    # ======================================================
    # Context Length
    # ======================================================

    remaining_tokens = CONTEXT_LENGTH - input_tokens

    usage_percent = (
        input_tokens / CONTEXT_LENGTH * 100
    )


    print()
    print("=" * 80)
    print("Context Length")
    print("=" * 80)

    print(f"入力トークン数 : {input_tokens}")
    print(f"Context Length : {CONTEXT_LENGTH}")
    print(f"残り          : {remaining_tokens}")
    print(f"使用率        : {usage_percent:.1f}%")


    # ======================================================
    # Qwen3-8B生成
    # ======================================================

    print()
    print("=" * 80)
    print("Qwen3-8B 回答生成")
    print("=" * 80)

    with torch.no_grad():

        output_ids = model.generate(
            **inputs,
            max_new_tokens=512,
            do_sample=False,
        )


    # ======================================================
    # 回答取得
    # ======================================================

    generated_ids = output_ids[0][input_tokens:]

    answer = tokenizer.decode(
        generated_ids,
        skip_special_tokens=True,
    )

    output_tokens = len(generated_ids)

    total_tokens = (
        input_tokens
        + output_tokens
    )


    # ======================================================
    # Token使用量
    # ======================================================

    print()
    print("=" * 80)
    print("Token使用量")
    print("=" * 80)

    print(f"入力トークン数 : {input_tokens}")
    print(f"生成トークン数 : {output_tokens}")
    print(f"合計Token数    : {total_tokens}")
    print(f"Context Length : {CONTEXT_LENGTH}")


    # ======================================================
    # Context使用率
    # ======================================================

    usage_percent_total = (
        total_tokens / CONTEXT_LENGTH * 100
    )

    remaining_tokens_total = (
        CONTEXT_LENGTH - total_tokens
    )

    print(f"Context残り    : {remaining_tokens_total}")
    print(f"Context使用率  : {usage_percent_total:.1f}%")


    # ======================================================
    # 回答
    # ======================================================

    print()
    print("=" * 80)
    print("Qwen3-8Bの回答")
    print("=" * 80)

    print(answer)

BGE-M3を読み込んでいます...


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

BGE-M3の読み込みが完了しました。


################################################################################
質問 1
################################################################################

質問: 北海道の65歳～69歳の人口は？

検索結果
[1] score=0.6676 北海道 女
[2] score=0.6583 北海道 計
[3] score=0.6579 北海道 男

Context Length
入力トークン数 : 1617
Context Length : 8192
残り          : 6575
使用率        : 19.7%

Qwen3-8B 回答生成

Token使用量
入力トークン数 : 1617
生成トークン数 : 240
合計Token数    : 1857
Context Length : 8192
Context残り    : 6335
Context使用率  : 22.7%

Qwen3-8Bの回答
<think>
Okay, let's tackle this question. The user is asking for the population of Hokkaido in the age group 65-69. 

First, I need to check the context provided. There are three search results. The user's question doesn't specify gender, so according to the rules, the gender should be "計" (total). 

Looking at the search results, the second one is for "計" (total) and includes the 65-69 age group with 327,278 people. The first result is for females only, which is 171,549

In [ ]:
# ==========================================
# Qwen RAG 失敗・境界条件テスト (enable_thinking=True)
# ==========================================

QUERIES = [
    "北海道の61歳～64歳の人口は？",
    "沖縄県の65歳～69歳の人口は？",
    "北海道の150歳～159歳の人口は？",
    "北海道の65歳～69歳の男女別人口は？",
    "北海道の人口は？",
]


# ==========================================================
# 質問ごとにRAG回答
# ==========================================================

for query_number, QUERY in enumerate(QUERIES, start=1):

    print()
    print()
    print("#" * 80)
    print(f"質問 {query_number}")
    print("#" * 80)

    print()
    print(f"質問: {QUERY}")


    # ======================================================
    # Query Embedding
    # ======================================================

    query_vector = embed_model.encode(
        QUERY,
        normalize_embeddings=True,
    )


    # ======================================================
    # Qdrant検索
    # ======================================================

    results = client.query_points(
        collection_name=COLLECTION_NAME,
        query=query_vector.tolist(),
        limit=TOP_K,
        with_payload=True,
    ).points


    # ======================================================
    # 検索結果
    # ======================================================

    print()
    print("=" * 80)
    print("検索結果")
    print("=" * 80)

    for i, result in enumerate(results, start=1):

        metadata = result.payload.get(
            "metadata",
            {}
        )

        print(
            f"[{i}] "
            f"score={result.score:.4f} "
            f"{metadata.get('都道府県名')} "
            f"{metadata.get('性別')}"
        )


    # ======================================================
    # Context作成
    # ======================================================

    contexts = []

    for i, result in enumerate(results, start=1):

        payload = result.payload
        metadata = payload.get("metadata", {})

        context = f"""
[検索結果 {i}]
都道府県名：{metadata.get('都道府県名')}
性別：{metadata.get('性別')}

{payload.get('text')}
"""

        contexts.append(context)


    context_text = "\n\n---\n\n".join(contexts)


    # ======================================================
    # User Prompt
    # ======================================================

    user_prompt = f"""
### 質問

{QUERY}

### コンテキスト

{context_text}

### 回答

質問に対して簡潔に回答してください。
"""


    # ======================================================
    # Messages
    # ======================================================

    messages = [
        {
            "role": "system",
            "content": system_prompt,
        },
        {
            "role": "user",
            "content": user_prompt,
        },
    ]


    # ======================================================
    # Chat Template
    # ======================================================

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=True,
    )


    # ======================================================
    # Tokenize
    # ======================================================

    inputs = tokenizer(
        text,
        return_tensors="pt",
    )

    inputs = {
        key: value.to(model.device)
        for key, value in inputs.items()
    }

    input_tokens = inputs["input_ids"].shape[1]


    # ======================================================
    # Context Length
    # ======================================================

    remaining_tokens = (
        CONTEXT_LENGTH - input_tokens
    )

    usage_percent = (
        input_tokens
        / CONTEXT_LENGTH
        * 100
    )

    print()
    print("=" * 80)
    print("Context Length")
    print("=" * 80)

    print(f"入力トークン数 : {input_tokens}")
    print(f"Context Length : {CONTEXT_LENGTH}")
    print(f"残り          : {remaining_tokens}")
    print(f"使用率        : {usage_percent:.1f}%")


    # ======================================================
    # Qwen3-8B生成
    # ======================================================

    print()
    print("=" * 80)
    print("Qwen3-8B 回答生成")
    print("=" * 80)

    with torch.no_grad():

        output_ids = model.generate(
            **inputs,
            max_new_tokens=512,
            do_sample=False,
        )


    # ======================================================
    # 回答取得
    # ======================================================

    generated_ids = output_ids[0][input_tokens:]

    answer = tokenizer.decode(
        generated_ids,
        skip_special_tokens=True,
    )

    output_tokens = len(generated_ids)

    total_tokens = (
        input_tokens
        + output_tokens
    )


    # ======================================================
    # Token使用量
    # ======================================================

    print()
    print("=" * 80)
    print("Token使用量")
    print("=" * 80)

    print(f"入力トークン数 : {input_tokens}")
    print(f"生成トークン数 : {output_tokens}")
    print(f"合計Token数    : {total_tokens}")
    print(f"Context Length : {CONTEXT_LENGTH}")

    remaining_tokens_total = (
        CONTEXT_LENGTH - total_tokens
    )

    usage_percent_total = (
        total_tokens
        / CONTEXT_LENGTH
        * 100
    )

    print(f"Context残り    : {remaining_tokens_total}")
    print(f"Context使用率  : {usage_percent_total:.1f}%")


    # ======================================================
    # 回答
    # ======================================================

    print()
    print("=" * 80)
    print("Qwen3-8Bの回答")
    print("=" * 80)

    print(answer)



################################################################################
質問 1
################################################################################

質問: 北海道の61歳～64歳の人口は？

検索結果
[1] score=0.6285 北海道 女
[2] score=0.6136 北海道 計
[3] score=0.6124 北海道 男

Context Length
入力トークン数 : 1621
Context Length : 8192
残り          : 6571
使用率        : 19.8%

Qwen3-8B 回答生成

Token使用量
入力トークン数 : 1621
生成トークン数 : 24
合計Token数    : 1645
Context Length : 8192
Context残り    : 6547
Context使用率  : 20.1%

Qwen3-8Bの回答
北海道の61歳～64歳の人口は177,945人です。


################################################################################
質問 2
################################################################################

質問: 沖縄県の65歳～69歳の人口は？

検索結果
[1] score=0.6880 沖縄県 男
[2] score=0.6878 沖縄県 女
[3] score=0.6857 沖縄県 計

Context Length
入力トークン数 : 1582
Context Length : 8192
残り          : 6610
使用率        : 19.3%

Qwen3-8B 回答生成

Token使用量
入力トークン数 : 1582
生成トークン数 : 20
合計Token数    : 1602
Context Length : 8192
Context残り    : 659